In [1]:
import torch
from torch import nn
from d2l import torch as d2l

In [2]:
from re import S


Size2D = int | tuple[int, int]

def to_pair(
    value: Size2D,
) -> tuple[int, int]:
    
    if isinstance(
        value,
        int,
    ):
        return (
            value,
            value,
        )
    
    return value


class PatchEmbedding(nn.Module):
    
    def __init__(
        self,
        image_size: Size2D = 96,
        patch_size: Size2D = 16,
        num_hiddens: int = 512,
    ) -> None:
        super().__init__()

        image_height, image_width = to_pair(
            image_size
        )        
        
        patch_height, patch_width = to_pair(
            patch_size
        )
        
        self.num_patches = (
            image_height // patch_height
        ) * (
            image_width // patch_width
        )
        
        self.conv = nn.LazyConv2d(
            out_channels=num_hiddens,
            kernel_size=(
                patch_height,
                patch_width,
            ),
            stride=(
                patch_height,
                patch_width,
            ),
        )
        
        
    def forward(
        self,
        X: torch.Tensor,
    ) -> torch.Tensor:
        
        # [B, C, H, W]
        # -> [B, D, H/P, W/P]
        X = self.conv(X)
        
        # -> [B, D, Number of Patches]
        X = X.flatten(
            start_dim=2
        )        
        
        # -> [B, Number of Patches, D]
        return X.transpose(
            1,
            2,
        )

In [3]:
img_size = 96
patch_size = 16
num_hiddens = 512
batch_size = 4


patch_embedding = PatchEmbedding(
    image_size=img_size,
    patch_size=patch_size,
    num_hiddens=num_hiddens,
)


# Dummy RGB Image
# [B, C, H, W]
images = torch.zeros(
    (
        batch_size,
        3,
        img_size,
        img_size,
    )
)


with torch.no_grad():
    # Convolution 직후의 Patch Grid
    patch_grid = patch_embedding.conv(
        images
    )
    
    # 최종 Patch Token Sequence
    patch_tokens = patch_embedding(
        images
    )


print(
    "Input images:",
    tuple(images.shape),
)

print(
    "Patch grid:",
    tuple(patch_grid.shape),
)

print(
    "Patch tokens:",
    tuple(patch_tokens.shape),
)

print(
    "Number of patches:",
    patch_embedding.num_patches,
)


expected_num_patches = (
    img_size // patch_size
) ** 2


d2l.check_shape(
    patch_grid,
    (
        batch_size,
        num_hiddens,
        img_size // patch_size,
        img_size // patch_size,
    ),
)

d2l.check_shape(
    patch_tokens,
    (
        batch_size,
        expected_num_patches,
        num_hiddens,
    ),
)

Input images: (4, 3, 96, 96)
Patch grid: (4, 512, 6, 6)
Patch tokens: (4, 36, 512)
Number of patches: 36
